# Stefan Problem

This is a testcase for evaporation.  
Some liquid evaporated at a planar fluid vapor interface.  
The evaporation is driven by heat transfer through the vapor, originating from a heated wall.  
Initially $t = 0$ there is no vapor in the system.  
The simulation is started from an analytical solution at $t_0 > 0$.  
We can then compare the simulated interface position in time to the analytically predicted.  
See `10.1016/j.ijheatmasstransfer.2021.121233`  

This worksheet also documents the results published in Rieckmann et al. (2024) `10.1016/j.jcp.2023.112716`

Note the following:
* A deviation from the source is the considerably larger surface tension, to deal with capillary timestep restrictions.
* If we initialize with zero velocity fields the interface movement is off in the first timestep. There are two workarounds: initialize with the analytical velocity (done now) or use a very small first timestep.
* Paraview seems to not read time data correcty. Thus time annotation is missing in videos.

In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using Microsoft.AspNetCore.Html;
Init();

In [ ]:
string proj = "StefanProblem";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.DefaultDatabase.Sessions;
sessions

## Grid and Boundary/Initial Value configuration

Rescale temperature, for some reason better for the solver:  
$\Theta = \frac{T - T_{s}}{T_{s} - T_{Wall}}$  
$\Delta T = {T_{s} - T_{Wall}}$  

$ \tilde{c} = \Delta T * c$  
$ \tilde{k} = \Delta T * k$  

In [ ]:
static class Constants{
    // careful order of declaration matters!!!
    public static double L = 0.001;
    public static double Xi = 0.066916063714767; // see Matlab Sheet StefanProblem.m

    // material parameters
    public static double rho_A = 958.4;
    public static double rho_B = 0.597;

    public static double mu_A = 2.8e-4;
    public static double mu_B = 1.26e-5;

    public static double Sigma = 0.059 / 1000000.0; // very small surface tension so that capillary timestep is bigger...

    // Original 373.15 K + 10 K => 0 + 1
    public static double c_A = 4216.0 * 10;
    public static double c_B = 2030.0 * 10;

    public static double k_A = 0.679 * 10;
    public static double k_B = 0.025 * 10;

    public static double hVap  = 2.258e6;
    public static double T_sat = 0.0;//373.15;
    public static double T_wall = 1.0;//T_sat + 10.0;

    public static double alpha_B = k_B / (c_B*rho_B);
    public static double eps = 1.0-rho_B/rho_A;

    public static double x0 = 0.000105;
    public static double t0 = Math.Pow(0.5 * x0 / Xi, 2) / alpha_B; // start time, to avoid singular massflux at t=0
    public static double tE = 0.6 - t0; // Endtime
    // capillary timestep , for finest res + highest degree, use this, for comparability?!, Is very small 1e-7 => 1e5 - 1e6 timesteps necessary => artificial surface tension?!
    public static Func<int, int, double> dt = (res, p) => 0.5 * Math.Sqrt((rho_A + rho_B) * Math.Pow(L / ((double)res * ((double)p + 1)), 3) / (2 * Math.PI * Math.Abs(Sigma)));
}

In [ ]:
Constants.t0

Effective grid width

In [ ]:
(Constants.L / ((64) * (4+1))) 

Maximum Velocity

In [ ]:
(Constants.eps * Constants.Xi * Constants.alpha_B / Math.Sqrt(Constants.alpha_B *  Constants.t0 ))

Capillary vs CFL Time-Step

In [ ]:
Constants.dt((int)(64), 4)

In [ ]:
(Constants.L / ((64) * (4+1)))  / (Constants.eps * Constants.Xi * Constants.alpha_B / Math.Sqrt(Constants.alpha_B *  Constants.t0 ))

In [ ]:
static class GridFactory{
    public static GridCommons Grid(int res, IDatabaseInfo myDb){
        double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(0, Constants.L, res + 1);
        double[] Ynodes = GenericBlas.Linspace(0, h, 2); // quasi 1D always use one cell
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

        grd.EdgeTagNames.Add(1, "freeslip_ZeroGradient");
        grd.EdgeTagNames.Add(2, "wall_ConstantTemperature");
        grd.EdgeTagNames.Add(3, "pressure_outlet_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if(Math.Abs(X[0]) < 1e-6)
                return 2;
            if(Math.Abs(X[0] -  Constants.L) < 1e-6)
                return 3;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }
}

In [ ]:
public static class BoundaryAndInitialValueFactory { 

    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
            stw.WriteLine("static class BoundaryAndInitialValues {");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfacePos(){");
            stw.WriteLine("         return (X,t) => 2 * " + Constants.Xi + " * Math.Sqrt(" + Constants.alpha_B + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfaceVel(){");
            stw.WriteLine("         return (X,t) => " + Constants.Xi + " * " + Constants.alpha_B + " / Math.Sqrt(" + Constants.alpha_B + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi(double[] X, double t){");
            stw.WriteLine("         return InterfacePos()(X,t) - X[0];");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureA(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_sat + "; // always saturation temp.");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double VelocityA(double[] X, double t){");
            stw.WriteLine("         return " + Constants.eps + " * InterfaceVel()(X,t);");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureB(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_wall + " + (" + Constants.T_sat + " - " + Constants.T_wall + ") / MathNet.Numerics.SpecialFunctions.Erf(" + Constants.Xi + ") * MathNet.Numerics.SpecialFunctions.Erf(X[0]/(2*Math.Sqrt(" + Constants.alpha_B + "*(t+" + Constants.t0 + "))));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureBBoundary(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_wall + ";");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureBInitial(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_wall + " - (" + Constants.T_wall + " - " + Constants.T_sat + ") * X[0]/InterfacePos()(X,t); // linear interpolation works better to project initial value");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double VelocityB(double[] X, double t){");
            stw.WriteLine("         return 0.0;");
            stw.WriteLine("     }");
            stw.WriteLine("}");
            return stw.ToString();
        }
    }
   
    static public Formula Get_VA() {
        return new Formula("BoundaryAndInitialValues.VelocityA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempA() {
        return new Formula("BoundaryAndInitialValues.TemperatureA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempB() {
        return new Formula("BoundaryAndInitialValues.TemperatureB", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempB_Boundary() {
        return new Formula("BoundaryAndInitialValues.TemperatureBBoundary", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempB_Initial() {
        return new Formula("BoundaryAndInitialValues.TemperatureBInitial", true, AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_Phi() {
        return new Formula("BoundaryAndInitialValues.Phi", true, AdditionalPrefixCode:GetPrefixCode());
    }
}

In [ ]:
var grd1D = Grid1D.LineGrid(GenericBlas.Linspace(0, Constants.x0, 100 + 1));
var b = new Basis(grd1D, 4);
var T_ini = new SinglePhaseField(b);
T_ini.ProjectField((MultidimensionalArray X, MultidimensionalArray R) => BoundaryAndInitialValueFactory.Get_TempB().EvaluateV(X, 0.0, R));
var gp = new Gnuplot();
gp.PlotField(T_ini);
gp.PlotNow()

## Initialize Controlfiles

In [ ]:
int[] resolutions = { 8, 16, 32, 64 };
int[] amr = { 0 };
int[] pDeg = { 1, 2, 3, 4 };

In [ ]:
List<XNSFE_Control> Controls = new List<XNSFE_Control>();

In [ ]:
foreach(int res in resolutions){
foreach(int p in pDeg){
    foreach(int lvl in amr){

        XNSFE_Control C = new XNSFE_Control();

        // basic database options
        // ======================
        C.DbPath      = Path.GetFullPath(BoSSSshell.WorkflowMgm.DefaultDatabase.Path);
        C.SessionName = proj + "_H" + res + "_AMR" + lvl + "_P" + p;
        C.ProjectName = proj;
        C.ProjectDescription = "stefan problem - testcase for evaporation";
        
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Degree", p));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("AMR", lvl));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Res", res));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Convtype", "H"));


        // DG degrees
        // ==========
        C.SetDGdegree(p);

        // Physical Parameters
        // ===================
        C.PhysicalParameters.Material = false;

        C.PhysicalParameters.IncludeConvection = true;
        C.ThermalParameters.IncludeConvection  = true;

        C.PhysicalParameters.rho_A = Constants.rho_A;
        C.PhysicalParameters.rho_B = Constants.rho_B;

        C.PhysicalParameters.mu_A = Constants.mu_A;
        C.PhysicalParameters.mu_B = Constants.mu_B;

        C.PhysicalParameters.Sigma = Constants.Sigma;

        C.ThermalParameters.rho_A = Constants.rho_A;
        C.ThermalParameters.rho_B = Constants.rho_B;

        C.ThermalParameters.c_A = Constants.c_A;
        C.ThermalParameters.c_B = Constants.c_B;

        C.ThermalParameters.k_A = Constants.k_A;
        C.ThermalParameters.k_B = Constants.k_B;

        C.ThermalParameters.hVap  = Constants.hVap;
        C.ThermalParameters.T_sat = Constants.T_sat;

        C.PhysicalParameters.betaS_A = 0.0;
        C.PhysicalParameters.betaS_B = 0.0;

        C.PhysicalParameters.betaL      = 0.0;
        C.PhysicalParameters.sliplength = 0.0;
        C.PhysicalParameters.theta_e    = Math.PI; 

        // grid generation
        // ===============
        var grd = GridFactory.Grid(res, BoSSSshell.WorkflowMgm.DefaultDatabase);
        C.SetGrid(grd);
        
        // Initial Values
        // ==============
        C.AddInitialValue("Phi", BoundaryAndInitialValueFactory.Get_Phi());
        C.AddInitialValue("Temperature#A", BoundaryAndInitialValueFactory.Get_TempA());
        C.AddInitialValue("Temperature#B", BoundaryAndInitialValueFactory.Get_TempB_Initial());
        C.AddInitialValue("VelocityX#A", BoundaryAndInitialValueFactory.Get_VA());

        // boundary conditions
        // ===================
        C.AddBoundaryValue("freeslip_ZeroGradient");
        C.AddBoundaryValue("pressure_outlet_ConstantTemperature", "Temperature#A", BoundaryAndInitialValueFactory.Get_TempA());
        C.AddBoundaryValue("wall_ConstantTemperature", "Temperature#B", BoundaryAndInitialValueFactory.Get_TempB_Boundary());

        // level set options
        // =================
        C.Option_LevelSetEvolution                        = BoSSS.Solution.LevelSetTools.LevelSetEvolution.StokesExtension;
        C.Timestepper_LevelSetHandling                    = BoSSS.Solution.XdgTimestepping.LevelSetHandling.LieSplitting;
        C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.LevelSetTools.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
        C.LSContiProjectionMethod                         = BoSSS.Solution.LevelSetTools.ContinuityProjectionOption.None;

        // Timestepping
        // ============
        C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
        C.dtFixed          = Constants.dt((int)(resolutions.Max() * Math.Pow(2, amr.Max())), pDeg.Max()); // Use same timestep for all, better comparability
        C.Endtime          = Constants.tE;
        C.NoOfTimesteps    = (int)Math.Ceiling(C.Endtime /C.dtFixed);
        C.saveperiod       = (int)Math.Floor(C.NoOfTimesteps / 300.0); //nice 30fps 10s videos
        C.rollingSaves     = false;

        // Nonlinear Solver
        C.NonLinearSolver.MaxSolverIterations = 25;
        C.FailOnSolverFail = false;
        
        // AMR
        // ============
        int level = lvl;
        C.AdaptiveMeshRefinement = lvl > 0;
        C.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRonNarrowband(){ maxRefinementLevel = level});
        C.AMR_startUpSweeps = lvl;

        // special options
        // ===============
        C.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.StefanProblemBenchmarkQuantities( 3 ) { LogPeriod = 4 }); // give the outlet edgetag

        Controls.Add(C);
    }
}
}

In [ ]:
int[] tRes = new[] {0, 1, 2, 3};

foreach(int t in tRes){
    int res = resolutions.Max();
    int lvl = amr.Max();
    int p = 3; //pDeg.Max();
    XNSFE_Control C = new XNSFE_Control();

    // basic database options
    // ======================
    C.DbPath      = Path.GetFullPath(BoSSSshell.WorkflowMgm.DefaultDatabase.Path);
    C.SessionName = proj + "_H" + res + "_AMR" + lvl + "_P" + p + "_T" + t;
    C.ProjectName = proj;
    C.ProjectDescription = "stefan problem - testcase for evaporation";
    
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Degree", p));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("AMR", lvl));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Res", res));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Convtype", "T"));


    // DG degrees
    // ==========
    C.SetDGdegree(p);

    // Physical Parameters
    // ===================
    C.PhysicalParameters.Material = false;

    C.PhysicalParameters.IncludeConvection = true;
    C.ThermalParameters.IncludeConvection  = true;

    C.PhysicalParameters.rho_A = Constants.rho_A;
    C.PhysicalParameters.rho_B = Constants.rho_B;

    C.PhysicalParameters.mu_A = Constants.mu_A;
    C.PhysicalParameters.mu_B = Constants.mu_B;

    C.PhysicalParameters.Sigma = Constants.Sigma;

    C.ThermalParameters.rho_A = Constants.rho_A;
    C.ThermalParameters.rho_B = Constants.rho_B;

    C.ThermalParameters.c_A = Constants.c_A;
    C.ThermalParameters.c_B = Constants.c_B;

    C.ThermalParameters.k_A = Constants.k_A;
    C.ThermalParameters.k_B = Constants.k_B;

    C.ThermalParameters.hVap  = Constants.hVap;
    C.ThermalParameters.T_sat = Constants.T_sat;

    C.PhysicalParameters.betaS_A = 0.0;
    C.PhysicalParameters.betaS_B = 0.0;

    C.PhysicalParameters.betaL      = 0.0;
    C.PhysicalParameters.sliplength = 0.0;
    C.PhysicalParameters.theta_e    = Math.PI; 

    // grid generation
    // ===============
    var grd = GridFactory.Grid(res, BoSSSshell.WorkflowMgm.DefaultDatabase);
    C.SetGrid(grd);
    
    // Initial Values
    // ==============
    C.AddInitialValue("Phi", BoundaryAndInitialValueFactory.Get_Phi());
    C.AddInitialValue("Temperature#A", BoundaryAndInitialValueFactory.Get_TempA());
    C.AddInitialValue("Temperature#B", BoundaryAndInitialValueFactory.Get_TempB_Initial());
    C.AddInitialValue("VelocityX#A", BoundaryAndInitialValueFactory.Get_VA());

    // boundary conditions
    // ===================
    C.AddBoundaryValue("freeslip_ZeroGradient");
    C.AddBoundaryValue("pressure_outlet_ConstantTemperature", "Temperature#A", BoundaryAndInitialValueFactory.Get_TempA());
    C.AddBoundaryValue("wall_ConstantTemperature", "Temperature#B", BoundaryAndInitialValueFactory.Get_TempB_Boundary());

    // level set options
    // =================
    C.Option_LevelSetEvolution                        = BoSSS.Solution.LevelSetTools.LevelSetEvolution.StokesExtension;
    C.Timestepper_LevelSetHandling                    = BoSSS.Solution.XdgTimestepping.LevelSetHandling.LieSplitting;
    C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.LevelSetTools.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
    C.LSContiProjectionMethod                         = BoSSS.Solution.LevelSetTools.ContinuityProjectionOption.None;

    // Timestepping
    // ============
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.dtFixed          = Constants.dt((int)(res * Math.Pow(2, amr.Max())), pDeg.Max())/Math.Pow(2, t);
    C.Endtime          = 50 * C.dtFixed * Math.Pow(2, t); //Constants.tE;
    C.NoOfTimesteps    = (int)Math.Ceiling(C.Endtime /C.dtFixed);
    C.saveperiod       = (int)Math.Floor(C.NoOfTimesteps / 300.0); //nice 30fps 10s videos
    C.rollingSaves     = false;
    
    // Nonlinear Solver
    C.NonLinearSolver.MaxSolverIterations = 25;
    C.FailOnSolverFail = false;

    // AMR
    // ============
    int level = lvl;
    C.AdaptiveMeshRefinement = lvl > 0;
    C.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRonNarrowband(){ maxRefinementLevel = level});
    C.AMR_startUpSweeps = lvl;

    // special options
    // ===============
    C.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.StefanProblemBenchmarkQuantities( 3 ) { LogPeriod = 1 }); // give the outlet edgetag

    Controls.Add(C);
}

In [ ]:
int ctrlCount = Controls.Count();
ctrlCount

In [ ]:
var jobs = Controls.Select(c => c.CreateJob()).ToArray();
jobs.ForEach(j => j.NumberOfThreads = 1);
jobs.ForEach(j => j.EnvironmentVars["BOSSS_ARG_7"] = j.NumberOfThreads.ToString());
jobs.Activate();

In [ ]:
BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(86400);

In [ ]:
int count = BoSSSshell.wmg.Sessions.Count();
int success = BoSSSshell.wmg.Sessions.Where(s => s.SuccessfulTermination).Count();

if(count != ctrlCount || count != success){
    throw new ApplicationException("Not all simulations calculated or finished successful!");
}